# Unpaired Multimodal Learning -- Colab Pipeline

**CS 5330 Pattern Recognition and Computer Vision -- Final Project**

Arul Agarwal, Anirudh Bakare, Utkarsh Milind Khursale

---

This notebook orchestrates the full project pipeline on a Google Colab GPU runtime: repository setup, data ingestion, hyperparameter tuning, model training, and latent-space evaluation.

## 1. Setup and Clone

Clone the project repository from GitHub and install all Python dependencies listed in `requirements.txt`.

In [ ]:
!git clone https://github.com/arulagarwal/CS5330-Final_Project.git
%cd CS5330-Final_Project
!pip install -r requirements.txt

## 2. Kaggle Authentication

Securely prompt for Kaggle API credentials and write them to a local `.env` file. This avoids hardcoding secrets in the notebook. You can obtain your API token from [kaggle.com/settings](https://www.kaggle.com/settings).

In [ ]:
import getpass

kaggle_username = getpass.getpass(prompt="Kaggle Username: ")
kaggle_key = getpass.getpass(prompt="Kaggle API Key: ")

with open(".env", "w") as f:
    f.write(f"KAGGLE_USERNAME={kaggle_username}\n")
    f.write(f"KAGGLE_KEY={kaggle_key}\n")

print("Credentials written to .env")

### Load Environment Variables

Read the `.env` file created above and inject the Kaggle credentials into the current process environment so that `kagglehub` can authenticate.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

assert os.environ.get("KAGGLE_USERNAME"), "KAGGLE_USERNAME not set"
assert os.environ.get("KAGGLE_KEY"), "KAGGLE_KEY not set"
print("Environment variables loaded successfully.")

## 3. Data Ingestion

Download the Stanford Cars dataset and generate synthetic unpaired text descriptions. This creates `data/images/` (196 class subfolders) and `data/text/` (196 CSV files with 5 descriptions each).

In [ ]:
!python download_data.py

In [ ]:
# Generate the Zero-Shot Text Anchors from the dataset
!python init_weights.py

## 4. Targeted Hyperparameter Tuning

Run a lightweight grid search over Learning Rate (`[1e-4, 5e-5]`) and Projection Dimension (`[256, 512]`). Each combination trains for up to 5 epochs with early stopping (patience of 2). The best parameters are saved to `best_params.txt`.

In [ ]:
!python tune.py

## 5. Final Model Training

Train the Unpaired Multimodal Learner for 20 epochs. The command below uses baseline parameters as a starting point. After reviewing the tuning results from Cell 4 (printed to console and saved in `best_params.txt`), substitute the `--lr` flag with the best learning rate identified by the grid search. The best model checkpoint is saved to `checkpoints/best_model.pt`, and the final top-1 test accuracy is reported at the end.

In [ ]:
# We will manually input the best parameters found in Cell 4 here.
# For now, executing with the baseline parameters and Early Stopping.
!python train.py --epochs 20 --batch-size 32 --lr 1e-4

## 6. Latent Space Evaluation

Load the frozen best checkpoint and extract 512-d shared-backbone embeddings for 5 selected car classes across both modalities. Reduce to 2-D with t-SNE and save the scatter plot to `latent_space.png`. Clustering of image and text points from the same class provides evidence of learned cross-modal synergies.

In [ ]:
!python test.py

---

### Display Latent Space Plot

Render the saved scatter plot inline for review.

In [ ]:
from IPython.display import Image, display
display(Image(filename="latent_space.png"))